# ResNet18 Property Condition Classification

This notebook implements ResNet18-based image classification for property condition assessment.

**Dataset:**
- Images with expert scores (1-5) as class labels
- Excludes portrait images (non-standard aspect ratios)
- 70/15/15 train/validation/test split

**Model:**
- ResNet18 pretrained on ImageNet
- Fine-tuned for 5-class property condition classification
- GPU support enabled


## Setup and Imports


In [ ]:
import os
import sys
import importlib
from pathlib import Path

# Add project root to Python path
if os.path.exists(os.path.join(os.getcwd(), 'src')):
    project_root = os.getcwd()
else:
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Force reload modules
if 'src.config' in sys.modules:
    importlib.reload(sys.modules['src.config'])
if 'src.data_loader' in sys.modules:
    importlib.reload(sys.modules['src.data_loader'])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from collections import Counter
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights

from src.data_loader import DataLoader as DataLoaderClass
from src.config import Config

# Set up plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Check GPU availability
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"🔧 Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA Version: {torch.version.cuda}")
else:
    print("   ⚠️ CUDA not available, using CPU (training will be slower)")


## Load Data and Filter Images


In [ ]:
# Load annotations
loader = DataLoaderClass()
df = loader.load_annotations()

# Filter images with valid expert scores
df = df[df['expert_score'].notna()].copy()
df['expert_score'] = df['expert_score'].astype(int)
df = df[df['expert_score'].between(1, 5)].copy()

print(f"📊 Total images with expert scores: {len(df)}")

# Filter out portrait images (non-standard aspect ratios)
# Keep only landscape images (width > height)
valid_images = []
for idx, row in df.iterrows():
    try:
        img_path = row['image_path']
        if os.path.exists(img_path):
            with Image.open(img_path) as img:
                width, height = img.size
                # Keep only landscape images (width >= height)
                if width >= height:
                    valid_images.append(idx)
    except Exception as e:
        continue

df_filtered = df.loc[valid_images].copy()
print(f"📊 Images after filtering portrait images: {len(df_filtered)}")
print(f"   Excluded: {len(df) - len(df_filtered)} portrait images")

# Class distribution
print("\n📊 Class Distribution:")
class_dist = df_filtered['expert_score'].value_counts().sort_index()
for score, count in class_dist.items():
    percentage = (count / len(df_filtered)) * 100
    print(f"   Score {score}: {count:4d} images ({percentage:5.2f}%)")

print(f"\n✅ Dataset ready: {len(df_filtered)} images")


## Create PyTorch Dataset Class


In [ ]:
class PropertyConditionDataset(Dataset):
    """Custom Dataset for property condition images"""
    
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img_path = row['image_path']
        label = row['expert_score'] - 1  # Convert to 0-4 range for PyTorch
        
        # Load image
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            # If image fails to load, return a black image
            image = Image.new('RGB', (224, 224), color='black')
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        return image, label

print("✅ Dataset class defined")


## Data Preprocessing


In [ ]:
# ImageNet normalization statistics
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Training transforms (no augmentation - same as validation/test)
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Validation/Test transforms (no augmentation)
val_test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

print("✅ Transforms defined")
print("   Training: no augmentation (using original images only)")
print("   Validation/Test: no augmentation")


## Create Data Splits and DataLoaders


In [ ]:
# Stratified split: 70% train, 15% val, 15% test
# First split: 70% train, 30% temp (val + test)
X = df_filtered[['image_path']]
y = df_filtered['expert_score']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Second split: 50% val, 50% test from temp (15% each of total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Create DataFrames for each split
train_df = pd.DataFrame({
    'image_path': X_train['image_path'].values,
    'expert_score': y_train.values
})
val_df = pd.DataFrame({
    'image_path': X_val['image_path'].values,
    'expert_score': y_val.values
})
test_df = pd.DataFrame({
    'image_path': X_test['image_path'].values,
    'expert_score': y_test.values
})

print("📊 Data Split:")
print(f"   Train: {len(train_df)} images ({len(train_df)/len(df_filtered)*100:.1f}%)")
print(f"   Validation: {len(val_df)} images ({len(val_df)/len(df_filtered)*100:.1f}%)")
print(f"   Test: {len(test_df)} images ({len(test_df)/len(df_filtered)*100:.1f}%)")

# Dataset size summary
print(f"\n📊 Final Dataset Sizes (after filtering portrait images):")
print(f"   Total filtered images: {len(df_filtered)}")
print(f"   Training set: {len(train_df)} images")
print(f"   Validation set: {len(val_df)} images")
print(f"   Test set: {len(test_df)} images")
print(f"\n💡 Note: No data augmentation is applied - using original images only.")

# Create datasets
train_dataset = PropertyConditionDataset(train_df, transform=train_transform)
val_dataset = PropertyConditionDataset(val_df, transform=val_test_transform)
test_dataset = PropertyConditionDataset(test_df, transform=val_test_transform)

# Create data loaders
batch_size = 16
num_workers = 4 if torch.cuda.is_available() else 0

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"\n✅ DataLoaders created")
print(f"   Batch size: {batch_size}")
print(f"   Workers: {num_workers}")


In [ ]:
def create_resnet18_model(num_classes=5, freeze_backbone=True, dropout_rate=0.5):
    """Create ResNet18 model with pretrained weights
    
    Args:
        num_classes: Number of output classes (default: 5)
        freeze_backbone: If True, freezes early layers to prevent overfitting
        dropout_rate: Dropout rate for final layer (default: 0.5)
    """
    # Load pretrained ResNet18
    model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    
    # Freeze early layers to prevent overfitting (transfer learning)
    if freeze_backbone:
        # Freeze: conv1, bn1, layer1, layer2, layer3
        # Keep trainable: layer4, fc
        for param in model.conv1.parameters():
            param.requires_grad = False
        for param in model.bn1.parameters():
            param.requires_grad = False
        for param in model.layer1.parameters():
            param.requires_grad = False
        for param in model.layer2.parameters():
            param.requires_grad = False
        for param in model.layer3.parameters():
            param.requires_grad = False
        # layer4 and fc remain trainable
    
    # Replace final fully connected layer with dropout
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout_rate),
        nn.Linear(num_features, num_classes)
    )
    
    return model

# Create model
freeze_backbone = True  # Freeze early layers to prevent overfitting
dropout_rate = 0.5  # Dropout rate for regularization
model = create_resnet18_model(num_classes=5, freeze_backbone=freeze_backbone, dropout_rate=dropout_rate)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print("✅ ResNet18 model created")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
print(f"   Frozen parameters: {frozen_params:,} ({frozen_params/total_params*100:.1f}%)")
if freeze_backbone:
    print(f"   ✅ Early layers frozen (conv1, bn1, layer1-3)")
print(f"   ✅ Dropout added to final layer (rate={dropout_rate})")

# Store class counts for later reporting
class_counts = df_filtered['expert_score'].value_counts().sort_index()
total_samples = len(df_filtered)

# Calculate class weights for imbalanced dataset (optional - can disable if hurting performance)
use_class_weights = True  # Set to False to disable class weights
if use_class_weights:
    class_weights = []
    for score in range(1, 6):
        count = class_counts.get(score, 0)
        if count > 0:
            # Balanced weighting: weight = total_samples / (num_classes * count)
            # Normalize so weights don't get too extreme
            weight = total_samples / (5 * count)
            # Cap weights to prevent extreme values (max 3x)
            weight = min(weight, 3.0)
            class_weights.append(weight)
        else:
            class_weights.append(1.0)
    
    class_weights_tensor = torch.FloatTensor(class_weights).to(device)
    print("\n📊 Class Weights (for imbalanced dataset - capped at 3x):")
    for i, weight in enumerate(class_weights):
        score = i + 1
        count = class_counts.get(score, 0)
        print(f"   Score {score}: {weight:.4f} (count: {count})")
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
else:
    print("\n📊 Using standard CrossEntropyLoss (no class weights)")
    criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print("\n✅ Training setup:")
if use_class_weights:
    print(f"   Loss function: Weighted CrossEntropyLoss (handles class imbalance)")
else:
    print(f"   Loss function: CrossEntropyLoss")
print(f"   Optimizer: Adam (lr=0.001)")
print(f"   Scheduler: ReduceLROnPlateau")


## Training Functions


In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, epoch=None):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # Progress bar
    pbar = tqdm(loader, desc=f'Epoch {epoch+1} [Train]' if epoch is not None else 'Training', 
                leave=False, ncols=100)
    
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        batch_total = labels.size(0)
        batch_correct = (predicted == labels).sum().item()
        total += batch_total
        correct += batch_correct
        
        # Update progress bar
        current_acc = 100 * correct / total
        current_loss = running_loss / (total / batch_total)
        pbar.set_postfix({'Loss': f'{current_loss:.4f}', 'Acc': f'{current_acc:.2f}%'})
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

def validate(model, loader, criterion, device, epoch=None):
    """Validate model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    # Progress bar
    pbar = tqdm(loader, desc=f'Epoch {epoch+1} [Val]' if epoch is not None else 'Validating', 
                leave=False, ncols=100)
    
    with torch.no_grad():
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            batch_total = labels.size(0)
            batch_correct = (predicted == labels).sum().item()
            total += batch_total
            correct += batch_correct
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            # Update progress bar
            current_acc = 100 * correct / total
            current_loss = running_loss / (total / batch_total)
            pbar.set_postfix({'Loss': f'{current_loss:.4f}', 'Acc': f'{current_acc:.2f}%'})
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc, all_preds, all_labels

print("✅ Training functions defined with progress bars")


## Training Loop


In [ ]:
# Training parameters
num_epochs = 50
best_val_loss = float('inf')  # Track best model for checkpointing

# History
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

print("🚀 Starting training...")
print("=" * 80)

# Main training loop with progress tracking
epoch_pbar = tqdm(range(num_epochs), desc="Training Progress", ncols=100)

for epoch in epoch_pbar:
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, epoch)
    
    # Validate
    val_loss, val_acc, _, _ = validate(model, val_loader, criterion, device, epoch)
    
    # Update learning rate
    scheduler.step(val_loss)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Update main progress bar
    epoch_pbar.set_postfix({
        'Train Loss': f'{train_loss:.4f}',
        'Train Acc': f'{train_acc:.2f}%',
        'Val Loss': f'{val_loss:.4f}',
        'Val Acc': f'{val_acc:.2f}%'
    })
    
    # Model checkpointing (save best model based on validation loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # Save best model
        model_path = os.path.join(project_root, 'models')
        os.makedirs(model_path, exist_ok=True)
        torch.save(model.state_dict(), os.path.join(model_path, 'resnet18_property_classifier.pth'))
        epoch_pbar.write(f"  ✅ Epoch {epoch+1}: Best model saved (val_loss: {val_loss:.4f})")

epoch_pbar.close()
print("\n✅ Training completed!")


## Load Best Model and Evaluate on Test Set


In [ ]:
# Load best model
model_path = os.path.join(project_root, 'models', 'resnet18_property_classifier.pth')
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path))
    print("✅ Loaded best model")
else:
    print("⚠️ Best model not found, using current model")

# Evaluate on test set
print("\n📊 Evaluating on test set...")
test_loss, test_acc, test_preds, test_labels = validate(model, test_loader, criterion, device)

print(f"\n{'='*80}")
print(f"Test Results:")
print(f"{'='*80}")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_acc:.2f}%")

# Classification report
print("\n📊 Classification Report:")
print(classification_report(
    test_labels, 
    test_preds, 
    target_names=[f'Score {i+1}' for i in range(5)]
))

# Confusion matrix (5-class)
cm = confusion_matrix(test_labels, test_preds)
print("\n📊 Confusion Matrix (5-class):")
print(cm)

# 3-class confusion matrix: (1,2) -> Low, 3 -> Medium, (4,5) -> High
def map_to_3class(score):
    """Map 5-class scores to 3-class: (1,2)->0, 3->1, (4,5)->2"""
    if score in [0, 1]:  # Scores 1-2 -> Low (class 0)
        return 0
    elif score == 2:  # Score 3 -> Medium (class 1)
        return 1
    else:  # Scores 4-5 -> High (class 2)
        return 2

test_labels_3class = [map_to_3class(label) for label in test_labels]
test_preds_3class = [map_to_3class(pred) for pred in test_preds]

cm_3class = confusion_matrix(test_labels_3class, test_preds_3class)
print("\n" + "="*80)
print("📊 3-Class Confusion Matrix:")
print("="*80)
print("Classes: Low (Scores 1-2) | Medium (Score 3) | High (Scores 4-5)")
print("\nConfusion Matrix:")
print(cm_3class)

# Print formatted confusion matrix
class_names_3class = ['Low (1-2)', 'Medium (3)', 'High (4-5)']
print("\nFormatted Confusion Matrix:")
print(f"{'':<15} {'Predicted':^45}")
print(f"{'Actual':<15} {'Low (1-2)':^15} {'Medium (3)':^15} {'High (4-5)':^15}")
print("-" * 60)
for i, row in enumerate(cm_3class):
    print(f"{class_names_3class[i]:<15} {row[0]:^15} {row[1]:^15} {row[2]:^15}")

# Calculate 3-class metrics
acc_3class = accuracy_score(test_labels_3class, test_preds_3class)
report_3class = classification_report(
    test_labels_3class,
    test_preds_3class,
    target_names=class_names_3class,
    output_dict=True
)

print("\n📊 3-Class Classification Report:")
print(classification_report(
    test_labels_3class,
    test_preds_3class,
    target_names=class_names_3class
))

print(f"\n3-Class Accuracy: {acc_3class*100:.2f}%")


## Visualizations


In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Validation Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history['train_acc'], label='Train Accuracy', marker='o')
axes[1].plot(history['val_acc'], label='Validation Accuracy', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Confusion matrix heatmap (5-class)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[f'Score {i+1}' for i in range(5)],
            yticklabels=[f'Score {i+1}' for i in range(5)])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Test Set (5-Class)')
plt.tight_layout()
plt.show()

# Confusion matrix heatmap (3-class)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_3class, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names_3class,
            yticklabels=class_names_3class)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Test Set (3-Class)')
plt.tight_layout()
plt.show()

# Print summary of imbalanced dataset handling
print("\n" + "="*80)
print("📊 Imbalanced Dataset Handling:")
print("="*80)
print("The dataset has imbalanced classes:")
for score, count in class_counts.items():
    percentage = (count / total_samples) * 100
    print(f"  Score {score}: {count:4d} images ({percentage:5.2f}%)")
print("\n✅ Solutions implemented:")
if use_class_weights and 'class_weights' in locals():
    print("  1. Class Weights in Loss Function:")
    print("     - Used weighted CrossEntropyLoss with balanced weighting (capped at 3x)")
    print("     - Penalizes misclassifications of minority classes more heavily")
    print("     - Class weights:", {f"Score {i+1}": f"{w:.4f}" for i, w in enumerate(class_weights)})
else:
    print("  1. Standard Loss Function:")
    print("     - Using CrossEntropyLoss without class weights")
print("\n  2. Stratified Data Splitting:")
print("     - Maintains class distribution across train/val/test splits")
print("     - Ensures all classes are represented in each split")
print("\n  3. No Data Augmentation:")
print("     - Using original images only (no augmentation applied)")
print("\n  4. Transfer Learning (Frozen Layers):")
print("     - Frozen: conv1, bn1, layer1, layer2, layer3 (pretrained ImageNet features)")
print("     - Trainable: layer4, fc (fine-tuned for property classification)")
print("     - Reduces overfitting by limiting trainable parameters")
print("\n  5. Dropout Regularization:")
print("     - Dropout rate: 0.5 in final fully connected layer")
print("     - Prevents overfitting by randomly disabling neurons during training")
print("\n  6. Full Training:")
print("     - Training runs for full 50 epochs (no early stopping)")
print("     - Best model (lowest validation loss) is saved during training")
print("="*80)

print("\n✅ Visualizations created")
